# 03 · Skill categories: one group per skill

This notebook gives every skill in `vacancy_skills` a **group** (Languages, Soft skills, Technical / IT, ...) and a **subgroup**. It uses the ESCO files in `data/reference/esco/` and your corrections in `data/reference/skill_category_overrides.csv`.

The result is a small lookup table, `data/processed/skill_categories.parquet`, with **one row per skill**. Notebooks 04 and 05 join it to `vacancy_skills` on `skill_uri`. It uses the tables from `02_clean_data.ipynb`.

## How it works

1. Join the skills in the data to ESCO `skills_en` on `skill_uri`.
2. A few old skills are no longer in ESCO v1.2.1. They are matched by name, first on the preferred name and then on alternative names.
3. Look up each skill in the ESCO lists (language, transversal, digital) and walk up the ESCO hierarchy to find its subgroup.
4. Apply the group rules below, in this order. The first rule that fits wins.
5. Apply your overrides. **Your corrections always win.**

| Order | Rule | Group |
|---|---|---|
| 1 | in the ESCO language list | Languages |
| 2 | in the digital list and under "working with computers" | Basic digital |
| 3 | in the digital list, anything else | Technical / IT |
| 4 | in the transversal list, or reuse level "transversal" | Soft skills |
| 5 | ESCO type "knowledge" | Knowledge |
| 6 | everything else | Job-specific skills |

Nothing in the data is changed. Run the cells from top to bottom.

## 0. Settings and loading

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

# The allowed groups. Overrides must use one of these names exactly.
GROUPS = ['Languages', 'Soft skills', 'Basic digital', 'Technical / IT',
          'Knowledge', 'Job-specific skills']

# Number of most-mentioned skills saved in the review file.
N_REVIEW = 150

# Support running from either the project root or notebooks/.
project_dir = Path.cwd()
if not (project_dir / 'data' / 'processed').is_dir():
    project_dir = project_dir.parent
processed_dir = project_dir / 'data' / 'processed'
reference_dir = project_dir / 'data' / 'reference'
esco_dir = reference_dir / 'esco'
overrides_path = reference_dir / 'skill_category_overrides.csv'
output_dir = project_dir / 'outputs' / '03_skill_categories'
output_dir.mkdir(parents=True, exist_ok=True)

def load_table(name):
    parquet = processed_dir / f'{name}.parquet'
    if parquet.exists():
        return pd.read_parquet(parquet)
    return pd.read_csv(processed_dir / f'{name}.csv.gz')

vacancies = load_table('vacancies')
vacancy_skills = load_table('vacancy_skills')
print('vacancies:', f'{len(vacancies):,}', 'rows')
print('vacancy_skills:', f'{len(vacancy_skills):,}', 'rows')

### ESCO files

- `skills_en`: one row per ESCO skill, with its type (skill or knowledge), reuse level and names. A few skills appear twice, so duplicates are dropped.
- The three lists: language, transversal and digital skills.
- `skillsHierarchy_en`: the top levels of the ESCO tree (level 0 to level 3).
- `broaderRelationsSkillPillar_en`: the parent of each skill. It links each skill up to the hierarchy.

In [ ]:
def read_esco(name):
    return pd.read_csv(esco_dir / f'{name}.csv', dtype=str)

esco = read_esco('skills_en')
print('skills_en rows:', f'{len(esco):,}', '| duplicate skills dropped:', esco['conceptUri'].duplicated().sum())
esco = esco.drop_duplicates('conceptUri').set_index('conceptUri')

language_list = set(read_esco('languageSkillsCollection_en')['conceptUri'])
transversal_list = set(read_esco('transversalSkillsCollection_en')['conceptUri'])
digital_list = set(read_esco('digitalSkillsCollection_en')['conceptUri'])
hierarchy = read_esco('skillsHierarchy_en')
broader = read_esco('broaderRelationsSkillPillar_en')

print('language list:', len(language_list), '| transversal list:', len(transversal_list),
      '| digital list:', len(digital_list))
print('hierarchy rows:', len(hierarchy), '| broader relations:', f'{len(broader):,}')

## 1. One row per skill

Count how often each skill appears: `mentions` is the number of vacancy–skill rows, `vacancies` the number of different vacancies.

In [ ]:
skills = (vacancy_skills
          .groupby('skill_uri', observed=True)
          .agg(skill=('skill', 'first'),
               mentions=('vacancy_key', 'size'),
               vacancies=('vacancy_key', 'nunique'))
          .reset_index())
skills['skill_uri'] = skills['skill_uri'].astype(str)
skills['skill'] = skills['skill'].astype(str)
skills = skills.sort_values('mentions', ascending=False, ignore_index=True)
print('distinct skills:', f'{len(skills):,}')
display(skills.head(10))

## 2. Join to ESCO

Most skills are found directly by `skill_uri`. The data covers 2015–2026 and ESCO has changed since then, so a few old URIs are no longer in ESCO v1.2.1. For those:

1. look for exactly one ESCO skill with the same **preferred name**;
2. otherwise look for exactly one ESCO skill with the same **alternative name**;
3. otherwise leave the skill unmatched. These are placed by hand in the overrides file.

The name comparison ignores upper and lower case.

In [ ]:
by_name = {}
for uri, label in esco['preferredLabel'].items():
    by_name.setdefault(str(label).strip().lower(), set()).add(uri)

by_alt_name = {}
for uri, labels in esco['altLabels'].dropna().items():
    for label in labels.split('\n'):
        if label.strip():
            by_alt_name.setdefault(label.strip().lower(), set()).add(uri)

def match_skill(uri, name):
    if uri in esco.index:
        return uri, 'uri'
    key = name.strip().lower()
    found = by_name.get(key, set())
    if len(found) == 1:
        return next(iter(found)), 'preferred name'
    found_alt = by_alt_name.get(key, set())
    if len(found) == 0 and len(found_alt) == 1:
        return next(iter(found_alt)), 'alternative name'
    if len(found) > 1 or len(found_alt) > 1:
        return None, 'ambiguous name'
    return None, 'no match'

matched = [match_skill(u, s) for u, s in zip(skills['skill_uri'], skills['skill'])]
skills['esco_uri'] = [m[0] for m in matched]
skills['matched_how'] = [m[1] for m in matched]

match_summary = (skills.groupby('matched_how')
                 .agg(skills=('skill_uri', 'size'), mentions=('mentions', 'sum'))
                 .sort_values('mentions', ascending=False))
match_summary['% of mentions'] = (100 * match_summary['mentions'] / skills['mentions'].sum()).round(2)
display(match_summary)

print('Old skills matched by name:')
display(skills[skills['matched_how'].isin(['preferred name', 'alternative name'])]
        .assign(esco_name=lambda d: d['esco_uri'].map(esco['preferredLabel']))
        [['skill', 'esco_name', 'matched_how', 'mentions']].head(15))
print('Old skills not matched (they need an override):')
display(skills[skills['esco_uri'].isna()][['skill', 'matched_how', 'mentions', 'skill_uri']])

## 3. ESCO details for each skill

For each matched skill, collect:

- `esco_skill_type`: "skill/competence" or "knowledge"
- `reuse_level`: transversal, cross-sector, sector-specific or occupation-specific
- whether it is in the language, transversal or digital list
- its **paths** up the ESCO tree. A skill can have several parents, so it can have several paths. For example, a skill can sit both under "skills" and under "knowledge".

In [ ]:
# Where each hierarchy concept sits: (level 0 term, level 1 term, level 2 term).
position = {}
for _, row in hierarchy.iterrows():
    for level in [3, 2, 1, 0]:
        uri = row[f'Level {level} URI']
        if isinstance(uri, str) and uri not in position:
            position[uri] = (row['Level 0 preferred term'],
                             row['Level 1 preferred term'] if level >= 1 else None,
                             row['Level 2 preferred term'] if level >= 2 else None)

parents = broader.groupby('conceptUri')['broaderUri'].apply(list).to_dict()

def paths_up(uri, seen=None):
    # Walk up the parents until reaching a concept in the hierarchy file.
    seen = set() if seen is None else seen
    if uri in position:
        return {position[uri]}
    found = set()
    for parent in parents.get(uri, []):
        if parent not in seen:
            seen.add(parent)
            found |= paths_up(parent, seen)
    return found

m = skills['esco_uri']
skills['esco_name'] = m.map(esco['preferredLabel'])
skills['esco_skill_type'] = m.map(esco['skillType'])
skills['reuse_level'] = m.map(esco['reuseLevel'])
skills['in_language_list'] = m.isin(language_list)
skills['in_transversal_list'] = m.isin(transversal_list)
skills['in_digital_list'] = m.isin(digital_list)
skills['paths'] = [sorted(paths_up(u), key=str) if isinstance(u, str) else [] for u in m]
skills['level1_terms'] = skills['paths'].map(lambda p: sorted({x[1] for x in p if x[1]}))

n_top = skills['paths'].map(lambda p: len({x[0] for x in p}))
print('skills with no path:', (skills['esco_uri'].notna() & (n_top == 0)).sum())
print('share of mentions with paths in more than one top level:',
      f"{100 * skills.loc[n_top > 1, 'mentions'].sum() / skills['mentions'].sum():.1f}%")
display(skills[['skill', 'esco_skill_type', 'reuse_level', 'in_language_list',
                'in_transversal_list', 'in_digital_list', 'level1_terms']].head(10))

## 4. Group rules

The rules from the table at the top, in order. Unmatched skills get "Unmatched" for now.

In [ ]:
def rule_group(row):
    if pd.isna(row['esco_uri']):
        return 'Unmatched'
    if row['in_language_list']:
        return 'Languages'
    if row['in_digital_list']:
        if 'working with computers' in row['level1_terms']:
            return 'Basic digital'
        return 'Technical / IT'
    if row['in_transversal_list'] or row['reuse_level'] == 'transversal':
        return 'Soft skills'
    if row['esco_skill_type'] == 'knowledge':
        return 'Knowledge'
    return 'Job-specific skills'

skills['rule_group'] = skills.apply(rule_group, axis=1)

def share_table(df, group_col):
    out = (df.groupby(group_col)
           .agg(skills=('skill_uri', 'size'), mentions=('mentions', 'sum'))
           .sort_values('mentions', ascending=False))
    out['% of mentions'] = (100 * out['mentions'] / out['mentions'].sum()).round(1)
    return out

print('Groups from the rules only:')
display(share_table(skills, 'rule_group'))

## 5. Your overrides

The file `data/reference/skill_category_overrides.csv` has one row per correction:

| column | meaning |
|---|---|
| `skill_uri` | the skill to correct, copied from `vacancy_skills` |
| `skill` | the skill name, to make the file readable (not used for matching) |
| `group` | the new group, one of the names in `GROUPS` |
| `subgroup` | optional. Leave it empty to use the ESCO subgroup |
| `reason` | why you changed it |

To add a correction, copy a skill's `skill_uri` from the review file (section 7) into a new row. Then run the notebook again.

The cell below checks that every group name is allowed, that every URI exists in the data, and that no skill appears twice.

In [ ]:
if overrides_path.exists():
    overrides = pd.read_csv(overrides_path, dtype=str, keep_default_na=False)
else:
    print('No overrides file found, using the rules only.')
    overrides = pd.DataFrame(columns=['skill_uri', 'skill', 'group', 'subgroup', 'reason'])
if 'subgroup' not in overrides.columns:
    overrides['subgroup'] = ''
overrides = overrides.apply(lambda col: col.str.strip())

bad_group = overrides[~overrides['group'].isin(GROUPS)]
unknown_uri = overrides[~overrides['skill_uri'].isin(skills['skill_uri'])]
repeated = overrides[overrides['skill_uri'].duplicated(keep=False)]
for title, problem in [('Group name not in GROUPS', bad_group),
                       ('skill_uri not found in vacancy_skills', unknown_uri),
                       ('Same skill_uri more than once', repeated)]:
    if len(problem):
        print(f'PROBLEM - {title}:')
        display(problem)
assert len(bad_group) == 0 and len(unknown_uri) == 0 and len(repeated) == 0, 'Fix the overrides file and run again.'

override_group = overrides.set_index('skill_uri')['group']
override_subgroup = overrides.set_index('skill_uri')['subgroup']

skills['group'] = skills['skill_uri'].map(override_group).fillna(skills['rule_group'])
skills['source'] = skills['skill_uri'].isin(override_group.index).map({True: 'override', False: 'rule'})

changed = skills[skills['source'] == 'override']
print(f'{len(overrides)} overrides applied, covering {changed["mentions"].sum():,} mentions.')
print('How the overrides move skills between groups (number of skills):')
display(pd.crosstab(changed['rule_group'], changed['group']))

## 6. Subgroups

The subgroup is the **level 1** term of the skill's ESCO path, for example "communication, collaboration and creativity" or "information and communication technologies (icts)". `subgroup_detail` is the level 2 term, when there is one.

A skill can have several paths, so the notebook picks the path that fits the group:

| group | path used |
|---|---|
| Languages | language skills and knowledge |
| Soft skills | transversal skills and competences |
| Knowledge, Technical / IT | knowledge |
| Job-specific skills, Basic digital | skills |

If there is no such path, it uses the first path in alphabetical order. A subgroup in your overrides file replaces this.

In [ ]:
PREFERRED_TOP = {
    'Languages': ['language skills and knowledge'],
    'Soft skills': ['transversal skills and competences', 'skills'],
    'Knowledge': ['knowledge'],
    'Technical / IT': ['knowledge', 'skills'],
    'Job-specific skills': ['skills', 'knowledge'],
    'Basic digital': ['skills', 'knowledge'],
}

def pick_path(paths, group):
    usable = [p for p in paths if p[1]]
    if not usable:
        return (None, None, None)
    for top in PREFERRED_TOP.get(group, []):
        for p in usable:
            if p[0] == top:
                return p
    return usable[0]

chosen = [pick_path(p, g) for p, g in zip(skills['paths'], skills['group'])]
skills['subgroup'] = [c[1] for c in chosen]
skills['subgroup_detail'] = [c[2] for c in chosen]

# Subgroups written in the overrides file win.
written = skills['skill_uri'].map(override_subgroup).replace('', pd.NA)
has_written = written.notna()
skills.loc[has_written, 'subgroup'] = written[has_written]
skills.loc[has_written, 'subgroup_detail'] = None
skills['subgroup'] = skills['subgroup'].fillna('other')

for g in GROUPS:
    top = (skills[skills['group'] == g].groupby('subgroup')['mentions'].sum()
           .sort_values(ascending=False).head(5))
    print(f'\n{g}:')
    for name, n in top.items():
        print(f'   {n:>9,}  {name}')

## 7. Review

The file `outputs/03_skill_categories/review_top_skills.csv` lists the most-mentioned skills with their group. Open it in Excel and read down the `group` column. If a skill is in the wrong group, add a row to the overrides file.

The 150 most-mentioned skills cover most of the mentions, so checking them fixes most of the picture.

In [ ]:
review_cols = ['skill', 'group', 'subgroup', 'source', 'rule_group', 'esco_skill_type',
               'reuse_level', 'mentions', 'vacancies', 'skill_uri']
review = skills.head(N_REVIEW)[review_cols]
review.to_csv(output_dir / 'review_top_skills.csv', index=False)
print(f'Top {N_REVIEW} skills cover',
      f"{100 * review['mentions'].sum() / skills['mentions'].sum():.1f}% of all mentions.")
print('Saved:', output_dir / 'review_top_skills.csv')

for g in GROUPS:
    names = skills.loc[skills['group'] == g, 'skill'].head(15).tolist()
    print(f'\n{g} - top 15:')
    print('   ' + ' | '.join(names))

## 8. Checks

In [ ]:
checks = {
    'one row per skill': skills['skill_uri'].is_unique,
    'every skill in vacancy_skills has a group': set(vacancy_skills['skill_uri'].astype(str)) == set(skills['skill_uri']),
    'every group is an allowed name (no Unmatched left)': skills['group'].isin(GROUPS).all(),
    'every skill has a subgroup': skills['subgroup'].notna().all(),
    'mentions add up to vacancy_skills rows': skills['mentions'].sum() == len(vacancy_skills),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)

left = skills[~skills['group'].isin(GROUPS)]
if len(left):
    print('Skills still without a group (add them to the overrides file):')
    display(left[['skill', 'mentions', 'skill_uri']])

## 9. Save the lookup table

Saved to `data/processed/skill_categories.parquet` (or `.csv.gz` if Parquet is not available), plus a plain CSV copy in `outputs/03_skill_categories/` that is easy to open.

In [ ]:
save_cols = ['skill_uri', 'skill', 'group', 'subgroup', 'subgroup_detail', 'esco_skill_type',
             'reuse_level', 'esco_uri', 'esco_name', 'matched_how', 'source', 'mentions', 'vacancies']
skill_categories = skills[save_cols].copy()

try:
    skill_categories.to_parquet(processed_dir / 'skill_categories.parquet', index=False)
    print('Saved:', processed_dir / 'skill_categories.parquet')
except ImportError:
    skill_categories.to_csv(processed_dir / 'skill_categories.csv.gz', index=False)
    print('Saved:', processed_dir / 'skill_categories.csv.gz')
skill_categories.to_csv(output_dir / 'skill_categories.csv', index=False)
print('Saved:', output_dir / 'skill_categories.csv')
print(f'{len(skill_categories):,} skills')

## 10. Summary and first look

Two views:

- **Share of mentions** per group: how all skill mentions split between the groups.
- **Share of vacancies** per group and year: the percentage of vacancies (complete months, at least one skill) that ask for at least one skill of the group. A vacancy can count in several groups, so these do not add up to 100%.

In [ ]:
summary = share_table(skill_categories, 'group')
summary['overrides'] = skill_categories.groupby('group')['source'].apply(lambda s: (s == 'override').sum())
display(summary)

vac = vacancies[vacancies['in_complete_period'] & vacancies['has_skills']][['vacancy_key', 'year']]
grouped = (vacancy_skills[['vacancy_key', 'skill_uri']]
           .assign(skill_uri=lambda d: d['skill_uri'].astype(str))
           .merge(skill_categories[['skill_uri', 'group']], on='skill_uri')
           .merge(vac, on='vacancy_key')
           .drop_duplicates(['vacancy_key', 'group']))
per_year = grouped.groupby(['year', 'group']).size().unstack('group')
vacancy_share = (100 * per_year.div(vac.groupby('year').size(), axis=0)).round(1)[GROUPS]
display(vacancy_share)
vacancy_share.to_csv(output_dir / 'vacancy_share_by_group_year.csv')

ax = vacancy_share.plot(figsize=(10, 5), marker='o')
ax.set_title('Share of vacancies asking for at least one skill of each group')
ax.set_ylabel('% of vacancies with skills')
ax.set_xlabel('year (2026 = January to August)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig(output_dir / 'vacancy_share_by_group_year.png', dpi=150)
plt.show()

## What to note down

- How the mentions split between the groups, and whether that matches what you expected
- Any skill in the review file that is in the wrong group. Add it to the overrides file and run again
- Which groups grow or shrink in the yearly chart. Compare 2015–2016 carefully, because a third of those vacancies have no skills recorded

Limits to keep in mind:

- The groups depend on ESCO's lists plus your choices. Another analyst could draw some borders differently, for example for management skills.
- ESCO lists a skill as "digital" if it involves computers, so some engineering or law topics (GDPR, electronics) fall into Technical / IT unless you move them. The overrides file already moves the most common ones.
- The yearly chart uses vacancy shares, so it is not affected by ads listing more skills over time.